# Warehouse AI Video Intelligence

Video → Gemini VLM → Structured Event JSON → Validation → Handling Rules → Grounded AI Assistant

This version:
- uses one fixed Gemini model
- disables SDK retries
- disables automatic function calling
- validates timestamps against the real video duration
- validates the complete event schema
- answers simple event-log questions locally from the logs
- prevents hallucination for unavailable metadata
- uses Gemini only for genuine reasoning questions

In [1]:
!pip install -q -U google-genai opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 13.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [2]:
import os
import json
import re
import time
import cv2

from google import genai
from google.genai import types

print("Imports ready.")

Imports ready.


In [3]:
GEMINI_MODEL = "gemini-3.1-flash-lite"

GEMINI_API_KEY = None

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    from getpass import getpass
    GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

if not GEMINI_API_KEY:
    raise RuntimeError("Gemini API key was not provided.")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(attempts=1)
    )
)

print("✅ Gemini client initialized")
print("✅ Fixed model:", GEMINI_MODEL)
print("✅ SDK retries: DISABLED")

✅ Gemini client initialized
✅ Fixed model: gemini-3.1-flash-lite
✅ SDK retries: DISABLED


## Upload warehouse video

In [4]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No video was uploaded.")

video_path = next(iter(uploaded.keys()))

print("✅ Video uploaded:", video_path)

Saving KD packets dragged, heavy box kept on other packets.mp4 to KD packets dragged, heavy box kept on other packets.mp4
✅ Video uploaded: KD packets dragged, heavy box kept on other packets.mp4


In [5]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Could not open the uploaded video.")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = frame_count / fps if fps and fps > 0 else 0.0

cap.release()

if duration <= 0:
    raise RuntimeError("Could not determine video duration.")

print("========== VIDEO INFO ==========")
print(f"FPS       : {fps:.2f}")
print(f"Frames    : {frame_count}")
print(f"Duration  : {duration:.2f} seconds")
print("================================")

========== VIDEO INFO ==========
FPS       : 30.00
Frames    : 1012
Duration  : 33.73 seconds


In [6]:
video_file = client.files.upload(file=video_path)

print("✅ Uploaded to Gemini")
print("File:", video_file.name)
print("Initial state:", video_file.state.name if video_file.state else "UNKNOWN")

✅ Uploaded to Gemini
File: files/zqyqfylei5f1
Initial state: PROCESSING


In [7]:
while not video_file.state or video_file.state.name != "ACTIVE":

    if video_file.state and video_file.state.name == "FAILED":
        raise RuntimeError("Gemini failed to process the uploaded video.")

    print("Processing video...")
    time.sleep(5)
    video_file = client.files.get(name=video_file.name)

print("✅ Video is ACTIVE and ready for analysis.")

Processing video...
✅ Video is ACTIVE and ready for analysis.


In [8]:
BEHAVIORS = [
    "product_dropped",
    "product_dragged",
    "product_thrown",
    "product_rolling",
    "rough_handling",
    "improper_stacking",
    "unstable_stacking",
    "product_outside_designated_area",
    "improper_equipment_usage",
    "unsafe_loading_sequence"
]

VALID_RISK = {"LOW", "MEDIUM", "HIGH", "CRITICAL"}
VALID_STATUS = {
    "observed_behaviour",
    "potential_risk",
    "confirmed_damage"
}

print("✅ 10 predefined behaviours loaded.")

✅ 10 predefined behaviours loaded.


## VLM structured JSON analysis

In [9]:
json_prompt = f"""
You are an AI Video Intelligence system for warehouse operations.

Analyze the uploaded warehouse video as a temporal sequence.

ACTUAL VIDEO DURATION:
{duration:.2f} seconds

TIMESTAMP REQUIREMENTS:
- Every start_time and end_time MUST be within 00:00 and {duration:.2f} seconds.
- Never invent timestamps outside the real video duration.
- Use only timestamps supported by the visible video sequence.

DETECT ONLY THESE PREDEFINED BEHAVIOURS:
{chr(10).join("- " + b for b in BEHAVIORS)}

IMPORTANT:
- Report only visually observable behaviour.
- Do not invent SKU, price, weight, injury, monetary value, identity, or other unavailable facts.
- Do not claim confirmed product damage unless actual damage is clearly visible.
- Distinguish observed behaviour, potential risk, and confirmed damage.
- Multiple behaviours may belong to the same event.
- If no predefined behaviour is visible, return an empty events array.

RETURN ONLY VALID JSON:

{{
  "video_id": "{os.path.basename(video_path)}",
  "video_duration_seconds": {duration:.2f},
  "video_summary": "short factual summary",
  "events": [
    {{
      "event_id": "EVT_001",
      "start_time": "00:00",
      "end_time": "00:00",
      "behavior": ["product_dragged"],
      "risk_level": "HIGH",
      "risk_score": 85,
      "evidence": "What is visibly happening",
      "potential_consequence": ["product_damage"],
      "recommended_action": "Corrective action",
      "confidence": 0.95,
      "status": "potential_risk"
    }}
  ]
}}

Constraints:
- risk_level: LOW, MEDIUM, HIGH, CRITICAL
- risk_score: 0-100
- confidence: 0-1
- status: observed_behaviour, potential_risk, confirmed_damage
- behavior must contain only predefined labels
- No Markdown
- No text outside JSON
"""
print("✅ VLM prompt prepared.")

✅ VLM prompt prepared.


In [10]:
analysis_response = None
analysis_error = None

try:
    analysis_response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[video_file, json_prompt],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.1,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
                disable=True
            ),
            http_options=types.HttpOptions(
                retry_options=types.HttpRetryOptions(attempts=1)
            )
        )
    )

    if not analysis_response.text:
        raise RuntimeError("Gemini returned an empty response.")

    print("✅ VLM analysis completed.")

except Exception as e:
    analysis_error = e
    print("❌ VLM analysis failed.")
    print(type(e).__name__)
    print(str(e))

✅ VLM analysis completed.


In [11]:
events_data = {
    "video_id": os.path.basename(video_path),
    "video_duration_seconds": round(duration, 2),
    "video_summary": "",
    "events": []
}

if analysis_response is not None:

    raw_output = analysis_response.text.strip()

    raw_output = re.sub(r"^```json\s*", "", raw_output, flags=re.IGNORECASE)
    raw_output = re.sub(r"^```\s*", "", raw_output)
    raw_output = re.sub(r"\s*```$", "", raw_output)

    try:
        events_data = json.loads(raw_output)
        print("✅ JSON parsed successfully.")
        print("Events detected:", len(events_data.get("events", [])))

    except json.JSONDecodeError as e:
        print("❌ JSON parsing failed.")
        print("Error:", e)
        print("\nRaw response:")
        print(analysis_response.text)

else:
    print("⚠️ No VLM result available.")
    print("events_data initialized with zero events.")

✅ JSON parsed successfully.
Events detected: 3


In [12]:
def timestamp_to_seconds(t):
    if not isinstance(t, str) or ":" not in t:
        raise ValueError(f"Invalid timestamp format: {t}")

    minutes, seconds = t.split(":", 1)
    return int(minutes) * 60 + float(seconds)


for event in events_data.get("events", []):

    required_fields = [
        "event_id", "start_time", "end_time", "behavior",
        "risk_level", "risk_score", "evidence",
        "potential_consequence", "recommended_action",
        "confidence", "status"
    ]

    for field in required_fields:
        assert field in event, (
            f"{event.get('event_id', 'UNKNOWN')}: missing '{field}'"
        )

    assert all(
        b in BEHAVIORS for b in event["behavior"]
    ), f"Unknown behaviour in {event['event_id']}"

    assert event["risk_level"] in VALID_RISK
    assert 0 <= float(event["risk_score"]) <= 100
    assert 0 <= float(event["confidence"]) <= 1
    assert event["status"] in VALID_STATUS

    start = timestamp_to_seconds(event["start_time"])
    end = timestamp_to_seconds(event["end_time"])

    assert 0 <= start <= duration, (
        f"{event['event_id']}: start timestamp outside video"
    )
    assert 0 <= end <= duration, (
        f"{event['event_id']}: end timestamp outside video"
    )
    assert end >= start, (
        f"{event['event_id']}: end_time before start_time"
    )

print("✅ Event validation passed.")

✅ Event validation passed.


In [13]:
print("\n" + "=" * 70)
print("DETECTED WAREHOUSE EVENTS")
print("=" * 70)

events = events_data.get("events", [])

if not events:
    print("No predefined risky behaviour was detected.")
else:
    for e in events:
        print(f"\nEvent ID     : {e['event_id']}")
        print(f"Time         : {e['start_time']} - {e['end_time']}")
        print(f"Behaviour    : {', '.join(e['behavior'])}")
        print(f"Risk         : {e['risk_level']} ({e['risk_score']}/100)")
        print(f"Confidence   : {e['confidence']}")
        print(f"Status       : {e['status']}")
        print(f"Evidence     : {e['evidence']}")
        print(f"Consequence  : {', '.join(e['potential_consequence'])}")
        print(f"Action       : {e['recommended_action']}")


DETECTED WAREHOUSE EVENTS

Event ID     : EVT_001
Time         : 00:00 - 00:06
Behaviour    : improper_equipment_usage
Risk         : MEDIUM (60/100)
Confidence   : 0.95
Status       : potential_risk
Evidence     : Workers are using a wooden pallet to slide packages across the floor instead of using a trolley or forklift.
Consequence  : product_damage, floor_damage
Action       : Use appropriate material handling equipment like a pallet jack or trolley.

Event ID     : EVT_002
Time         : 00:07 - 00:14
Behaviour    : improper_stacking, unstable_stacking
Risk         : HIGH (80/100)
Confidence   : 0.98
Status       : potential_risk
Evidence     : A heavy box is placed on top of smaller, potentially fragile packets.
Consequence  : product_damage, stack_collapse
Action       : Ensure heavy items are placed at the bottom of the stack.

Event ID     : EVT_003
Time         : 00:15 - 00:29
Behaviour    : product_dragged
Risk         : HIGH (85/100)
Confidence   : 0.95
Status       : obser

In [14]:
handling_rules = {
    "product_dropped": ("WH-001", "Products must be lifted and placed gently; never throw or drop packages."),
    "product_dragged": ("WH-002", "Use suitable handling equipment instead of dragging products."),
    "product_thrown": ("WH-003", "Products must be handled carefully and placed in a controlled manner."),
    "product_rolling": ("WH-004", "Use appropriate material-handling equipment; do not roll unless designed for it."),
    "rough_handling": ("WH-005", "Handle every product carefully and in a controlled manner."),
    "improper_stacking": ("WH-006", "Stack heavier products at the bottom and lighter products on top."),
    "unstable_stacking": ("WH-007", "Load products in a stable configuration."),
    "product_outside_designated_area": ("WH-008", "Stage products systematically in the designated area."),
    "improper_equipment_usage": ("WH-009", "Use the correct equipment for material movement."),
    "unsafe_loading_sequence": ("WH-010", "Load products in a stable and planned sequence.")
}

for event in events_data.get("events", []):
    event["handling_rules"] = []

    for behavior in event.get("behavior", []):
        if behavior in handling_rules:
            rule_id, rule_text = handling_rules[behavior]
            event["handling_rules"].append({
                "rule_id": rule_id,
                "rule": rule_text
            })

with open("warehouse_event_log.json", "w") as f:
    json.dump(events_data, f, indent=2)

print("✅ Handling rules attached and event log saved.")

✅ Handling rules attached and event log saved.


In [15]:
def direct_event_answer(question, data):

    q = question.lower().strip()
    events = data.get("events", [])

    if any(x in q for x in [
        "how many events", "number of events",
        "total events", "count of events"
    ]):
        return f"There are {len(events)} detected event(s) in the available event log."

    if any(x in q for x in [
        "what risky behaviours", "which risky behaviours",
        "what behaviours were detected", "which behaviours were detected"
    ]):
        behaviours = []
        for e in events:
            for b in e.get("behavior", []):
                if b not in behaviours:
                    behaviours.append(b)

        if not behaviours:
            return "No predefined risky behaviours were detected."

        return "Detected risky behaviours:\n\n" + "\n".join(
            f"• {b}" for b in behaviours
        )

    if "high risk" in q or "high-risk" in q:
        high = [
            e for e in events
            if e.get("risk_level", "").upper() == "HIGH"
        ]

        if not high:
            return "No HIGH-risk events were found in the available event logs."

        return "HIGH-risk events:\n\n" + "\n\n".join(
            f"• {e['event_id']}\n"
            f"  Timestamp: {e['start_time']} - {e['end_time']}\n"
            f"  Behaviour: {', '.join(e['behavior'])}\n"
            f"  Risk Score: {e['risk_score']}"
            for e in high
        )

    if any(x in q for x in [
        "highest risk", "highest-risk",
        "highest risk score", "most risky event"
    ]):
        if not events:
            return "No events are available."

        e = max(events, key=lambda x: float(x.get("risk_score", 0)))

        return (
            f"The highest-risk event is {e['event_id']}.\n\n"
            f"Timestamp: {e['start_time']} - {e['end_time']}\n"
            f"Behaviour: {', '.join(e['behavior'])}\n"
            f"Risk Level: {e['risk_level']}\n"
            f"Risk Score: {e['risk_score']}\n"
            f"Confidence: {e['confidence']}"
        )

    if "timeline" in q or "all incidents" in q:
        if not events:
            return "No incidents were detected."

        ordered = sorted(
            events,
            key=lambda x: timestamp_to_seconds(x["start_time"])
        )

        return "Incident timeline:\n\n" + "\n".join(
            f"• {e['event_id']}: {e['start_time']} - {e['end_time']} → "
            f"{', '.join(e['behavior'])}"
            for e in ordered
        )

    match = re.search(r"\bEVT[_-]?\d+\b", question, re.IGNORECASE)

    if match:
        wanted = match.group(0).replace("-", "_").upper()

        for e in events:
            if e.get("event_id", "").upper() == wanted:

                rules = "\n".join(
                    f"• {r['rule_id']}: {r['rule']}"
                    for r in e.get("handling_rules", [])
                ) or "No mapped handling rule available."

                return (
                    f"Event {wanted}\n\n"
                    f"Timestamp: {e.get('start_time')} - {e.get('end_time')}\n"
                    f"Behaviour: {', '.join(e.get('behavior', []))}\n"
                    f"Risk Level: {e.get('risk_level')}\n"
                    f"Risk Score: {e.get('risk_score')}\n"
                    f"Confidence: {e.get('confidence')}\n"
                    f"Status: {e.get('status')}\n\n"
                    f"Evidence:\n{e.get('evidence', 'Not available')}\n\n"
                    f"Potential Consequence:\n"
                    f"{', '.join(e.get('potential_consequence', []))}\n\n"
                    f"Recommended Action:\n"
                    f"{e.get('recommended_action', 'Not available')}\n\n"
                    f"Handling Rules:\n{rules}"
                )

        return f"I could not find {wanted} in the available event logs."

    return None


print("✅ Local JSON assistant ready.")

✅ Local JSON assistant ready.


In [16]:
UNAVAILABLE_PATTERNS = [
    "monetary value", "product value", "price",
    "cost of the product", "sku", "product sku",
    "exact weight", "product weight",
    "who was responsible", "responsible for the incident",
    "who caused", "was anyone injured", "anyone injured",
    "injury", "drop height", "exact drop height",
    "financial loss", "loss amount"
]

def is_unavailable_question(question):
    q = question.lower()
    return any(pattern in q for pattern in UNAVAILABLE_PATTERNS)

print("✅ Hallucination protection ready.")

✅ Hallucination protection ready.


In [17]:
def gemini_reasoning_answer(question, data):

    context = json.dumps(data, indent=2)

    prompt = f"""
You are an AI Warehouse Operations Assistant.

Use ONLY the warehouse event data below.

WAREHOUSE EVENT DATA:
{context}

USER QUESTION:
{question}

Rules:
- Never invent facts.
- Never invent SKU, weight, price, monetary value, injury, identity,
  exact measurements, or facts not present in the event data.
- Distinguish observed behaviour, potential risk, and confirmed damage.
- Confirmed damage may only be stated if the event status is confirmed_damage.
- If requested information is not present, say exactly:
  "I could not find this information in the available event logs."
- Be concise and operationally useful.
"""

    try:
        result = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=True
                ),
                http_options=types.HttpOptions(
                    retry_options=types.HttpRetryOptions(attempts=1)
                )
            )
        )

        return result.text if result.text else "No answer was returned."

    except Exception as e:
        error_text = str(e)

        if "503" in error_text or "UNAVAILABLE" in error_text:
            return (
                "Gemini is temporarily unavailable. "
                "JSON-based event questions still work locally."
            )

        if "429" in error_text or "RESOURCE_EXHAUSTED" in error_text:
            return (
                "Gemini API quota/rate limit was reached. "
                "JSON-based event questions still work locally."
            )

        return f"Gemini reasoning request failed: {error_text}"

In [18]:
def warehouse_assistant(question, data):

    # First: answer from local event data.
    direct = direct_event_answer(question, data)

    if direct is not None:
        return direct

    # Second: never ask Gemini for known-unavailable metadata.
    if is_unavailable_question(question):
        return "I could not find this information in the available event logs."

    # Third: use Gemini only for reasoning.
    return gemini_reasoning_answer(question, data)


print("✅ Final warehouse assistant ready.")

✅ Final warehouse assistant ready.


## Interactive assistant

Type `exit` to stop.

In [19]:
print("=" * 70)
print("🤖 WAREHOUSE AI OPERATIONS ASSISTANT")
print("=" * 70)
print("Ask questions about the analyzed warehouse video.")
print("Type 'exit' to stop.")
print("=" * 70)

while True:

    question = input("\nYou: ").strip()

    if not question:
        continue

    if question.lower() in ["exit", "quit", "bye"]:
        print("Assistant: Goodbye! 👋")
        break

    answer = warehouse_assistant(question, events_data)

    print("\nAssistant:", answer)

🤖 WAREHOUSE AI OPERATIONS ASSISTANT
Ask questions about the analyzed warehouse video.
Type 'exit' to stop.

You: Why was EVT_001 classified as HIGH risk?

Assistant: HIGH-risk events:

• EVT_002
  Timestamp: 00:07 - 00:14
  Behaviour: improper_stacking, unstable_stacking
  Risk Score: 80

• EVT_003
  Timestamp: 00:15 - 00:29
  Behaviour: product_dragged
  Risk Score: 85

You: What visual evidence supports this classification? How could EVT_001 have been prevented? What should the warehouse supervisor do? Did actual product damage occur or was it only a potential risk?

Assistant: Event EVT_001

Timestamp: 00:00 - 00:06
Behaviour: improper_equipment_usage
Risk Level: MEDIUM
Risk Score: 60
Confidence: 0.95
Status: potential_risk

Evidence:
Workers are using a wooden pallet to slide packages across the floor instead of using a trolley or forklift.

Potential Consequence:
product_damage, floor_damage

Recommended Action:
Use appropriate material handling equipment like a pallet jack or tro

## Demo questions

### Local — no Gemini call
- What risky behaviours were detected in this video?
- How many events were detected?
- Which events are HIGH risk?
- Which event has the highest risk score?
- Give me the timeline of all incidents.
- Tell me everything about EVT_001.

### Gemini reasoning
- Why was EVT_001 classified as HIGH risk?
- What visual evidence supports this classification?
- How could EVT_001 have been prevented?
- What should the warehouse supervisor do?
- Did actual product damage occur or was it only a potential risk?

### Grounding tests
- What was the monetary value of the product?
- What was the exact weight of the product?
- What was the product SKU?
- Was anyone injured?

Expected unavailable-data response:
**I could not find this information in the available event logs.**